# Comprehensive EDA of Brazilian Human Rights Hotline Complaints

## Step 5: Bivariate Analysis

This notebook examines relationships between pairs of variables in the cleaned Disque 100-style administrative dataset. It combines normalized cross-tabulations, interactive Plotly charts, association tests, effect sizes, standardized residuals, sparse-cell diagnostics, and false-discovery-rate correction.

### Main questions

- Do recorded violation categories vary by victim age, gender, race/color, or vulnerable group?
- Does the victim-suspect relationship vary across violation categories?
- Is emergency classification associated with violation type or complaint context?
- Do recorded profiles vary geographically?
- Does the number of victims differ across categorical groups?

> These analyses describe **associations among recorded hotline observations**. They do not establish causality, population prevalence, or legal responsibility.


## 5.1 Imports and analytical settings

The notebook defaults to record-level analysis. Use complaint mode only after confirming that selected variables are stable within `hash` or after constructing a documented complaint-level table.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

PARQUET_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.parquet")
CSV_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.csv")
OUTPUT_DIR = Path("outputs/step_05_bivariate_analysis")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COUNTING_MODE = "record"  # record or complaint
IDENTIFIER = "hash"
ALPHA = 0.05
MIN_EXPECTED = 5
MAX_LEVELS = 15
TOP_N = 12
EXPORT_FIGURES_HTML = True
PLOTLY_TEMPLATE = "plotly_white"


## 5.2 Load the cleaned dataset and define the analytical population


In [2]:
def load_clean_dataset():
    if PARQUET_PATH.exists():
        return pd.read_parquet(PARQUET_PATH), PARQUET_PATH
    if CSV_PATH.exists():
        return pd.read_csv(CSV_PATH, low_memory=False), CSV_PATH
    raise FileNotFoundError("Run Step 3 or update PARQUET_PATH/CSV_PATH.")


df, DATA_SOURCE = load_clean_dataset()
if COUNTING_MODE not in {"record", "complaint"}:
    raise ValueError("COUNTING_MODE must be 'record' or 'complaint'.")
if COUNTING_MODE == "complaint":
    if IDENTIFIER not in df.columns:
        raise ValueError("Complaint mode requires hash.")
    df_analysis = df.dropna(subset=[IDENTIFIER]).drop_duplicates(IDENTIFIER).copy()
    unit_label = "complaints"
else:
    df_analysis = df.copy()
    unit_label = "records"

print("Source:", DATA_SOURCE.resolve())
print(f"Analytical population: {len(df_analysis):,} {unit_label}")


Source: C:\Projetos\eda-disque100-completo\eda\outputs\step_03_data_cleaning\disque100_clean.parquet
Analytical population: 8,401 records


## 5.3 Planned variable pairs

The plan prioritizes substantively meaningful relationships and avoids indiscriminate testing of every possible pair. Pairs unavailable in the dataset are skipped automatically.


In [3]:
PAIR_PLAN = [
    ("violacao", "faixa_etaria_da_vitima", "Violation by victim age group"),
    ("violacao", "genero_da_vitima", "Violation by victim gender"),
    ("violacao", "raca_cor_da_vitima", "Violation by victim race/color"),
    ("violacao", "grupo_vulneravel", "Violation by vulnerable group"),
    ("violacao", "relacao_vitima_suspeito", "Violation by victim-suspect relationship"),
    ("violacao", "denuncia_emergencial", "Violation by emergency classification"),
    ("violacao", "uf", "Violation by complaint UF"),
    ("grupo_vulneravel", "faixa_etaria_da_vitima", "Vulnerable group by victim age"),
    ("relacao_vitima_suspeito", "genero_da_vitima", "Relationship by victim gender"),
    ("relacao_vitima_suspeito", "faixa_etaria_da_vitima", "Relationship by victim age"),
    ("genero_da_vitima", "genero_do_suspeito", "Victim gender by suspect gender"),
    ("faixa_etaria_da_vitima", "faixa_etaria_do_suspeito", "Victim age by suspect age"),
    ("denuncia_emergencial", "cenario_da_violacao", "Emergency status by setting"),
    ("denuncia_emergencial", "frequencia", "Emergency status by reported frequency"),
    ("canal_de_atendimento", "denuncia_emergencial", "Channel by emergency status"),
]

available_pairs = [(x, y, title) for x, y, title in PAIR_PLAN if x in df_analysis.columns and y in df_analysis.columns]
skipped_pairs = [(x, y, title) for x, y, title in PAIR_PLAN if x not in df_analysis.columns or y not in df_analysis.columns]
display(pd.DataFrame(available_pairs, columns=["row_variable", "column_variable", "title"]))
display(pd.DataFrame(skipped_pairs, columns=["row_variable", "column_variable", "title"]))


,row_variable,column_variable,title
0,violacao,faixa_etaria_da_vitima,Violation by victim age group
1,violacao,genero_da_vitima,Violation by victim gender
2,violacao,raca_cor_da_vitima,Violation by victim race/color
3,violacao,grupo_vulneravel,Violation by vulnerable group
4,violacao,relacao_vitima_suspeito,Violation by victim-suspect relationship
5,violacao,denuncia_emergencial,Violation by emergency classification
6,violacao,uf,Violation by complaint UF
7,grupo_vulneravel,faixa_etaria_da_vitima,Vulnerable group by victim age
8,relacao_vitima_suspeito,genero_da_vitima,Relationship by victim gender
9,relacao_vitima_suspeito,faixa_etaria_da_vitima,Relationship by victim age


,row_variable,column_variable,title


## 5.4 Helper functions

For categorical pairs, the notebook reports:

- observed counts
- row percentages
- column percentages
- Pearson chi-square test
- bias-corrected Cramér's V
- expected-cell diagnostics
- Pearson standardized residuals

A chi-square p-value is not emphasized when more than 20% of expected cells are below 5 or any expected cell is below 1. In those cases, category aggregation based on the official hierarchy or an exact/Monte Carlo procedure should be considered.


In [4]:
def top_levels(series, n=TOP_N):
    return set(series.dropna().value_counts().head(n).index)


def prepare_pair(data, x, y, max_levels=MAX_LEVELS):
    pair = data[[x, y]].dropna().copy()
    aggregation = []
    for col in [x, y]:
        if pair[col].nunique() > max_levels:
            keep = top_levels(pair[col], max_levels - 1)
            pair[col] = pair[col].where(pair[col].isin(keep), "OTHER")
            aggregation.append(col)
    return pair, aggregation


def cramers_v_bias_corrected(table):
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    if n <= 1:
        return np.nan
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    denominator = min(kcorr-1, rcorr-1)
    return np.sqrt(phi2corr/denominator) if denominator > 0 else np.nan


def categorical_association(data, x, y):
    pair, aggregated = prepare_pair(data, x, y)
    table = pd.crosstab(pair[x], pair[y])
    if table.shape[0] < 2 or table.shape[1] < 2:
        return None
    chi2, p, dof, expected = chi2_contingency(table, correction=False)
    expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)
    residuals = (table - expected_df) / np.sqrt(expected_df)
    low5 = int((expected < MIN_EXPECTED).sum())
    low1 = int((expected < 1).sum())
    cells = expected.size
    diagnostics = {
        "x": x, "y": y, "n_complete": int(table.to_numpy().sum()),
        "rows": table.shape[0], "columns": table.shape[1],
        "chi2": chi2, "dof": dof, "p_value": p,
        "cramers_v": cramers_v_bias_corrected(table),
        "expected_below_5": low5, "expected_below_5_pct": round(low5/cells*100, 2),
        "expected_below_1": low1,
        "assumptions_ok": low1 == 0 and low5/cells <= 0.20,
        "aggregated_to_other": ", ".join(aggregated)
    }
    return table, expected_df, residuals, diagnostics


def save_figure(fig, filename):
    if EXPORT_FIGURES_HTML:
        fig.write_html(FIGURE_DIR / filename, include_plotlyjs=True)


def safe_filename(text):
    return "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in text.lower())


## 5.5 Run categorical association analyses

Row-normalized heatmaps answer: “Within each row category, how are observations distributed across column categories?” Count heatmaps remain available in the exported tables.


In [5]:
association_rows = []
residual_rows = []

for x, y, title in available_pairs:
    result = categorical_association(df_analysis, x, y)
    if result is None:
        association_rows.append({"x": x, "y": y, "title": title, "status": "insufficient_variation"})
        continue

    counts, expected, residuals, diagnostics = result
    diagnostics.update({"title": title, "status": "completed"})
    association_rows.append(diagnostics)

    row_pct = counts.div(counts.sum(axis=1), axis=0) * 100
    col_pct = counts.div(counts.sum(axis=0), axis=1) * 100
    stem = safe_filename(f"{x}_by_{y}")
    counts.to_csv(TABLE_DIR / f"{stem}_counts.csv", encoding="utf-8-sig")
    row_pct.to_csv(TABLE_DIR / f"{stem}_row_percent.csv", encoding="utf-8-sig")
    col_pct.to_csv(TABLE_DIR / f"{stem}_column_percent.csv", encoding="utf-8-sig")
    expected.to_csv(TABLE_DIR / f"{stem}_expected.csv", encoding="utf-8-sig")
    residuals.to_csv(TABLE_DIR / f"{stem}_standardized_residuals.csv", encoding="utf-8-sig")

    fig = px.imshow(
        row_pct.round(1), text_auto=".1f", aspect="auto",
        color_continuous_scale="Blues", zmin=0,
        title=f"{title}: row percentages",
        labels={"color": "Row %", "x": y, "y": x}
    )
    fig.update_layout(template=PLOTLY_TEMPLATE, height=max(480, 30*len(row_pct)+180))
    fig.show()
    save_figure(fig, f"{stem}_row_percent.html")

    fig_res = px.imshow(
        residuals.round(2), text_auto=".2f", aspect="auto",
        color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
        title=f"{title}: standardized residuals",
        labels={"color": "Residual", "x": y, "y": x}
    )
    fig_res.add_annotation(
        text="|residual| ≥ 1.96 indicates cells contributing strongly to the association",
        xref="paper", yref="paper", x=0, y=-0.18, showarrow=False
    )
    fig_res.update_layout(template=PLOTLY_TEMPLATE, height=max(500, 30*len(residuals)+200))
    fig_res.show()
    save_figure(fig_res, f"{stem}_residuals.html")

    stacked = row_pct.reset_index().melt(id_vars=x, var_name=y, value_name="percent")
    fig_stack = px.bar(
        stacked, x=x, y="percent", color=y, barmode="stack",
        title=f"{title}: composition within {x}",
        labels={"percent": "Percentage within row"},
        color_discrete_sequence=px.colors.qualitative.Safe
    )
    fig_stack.update_layout(template=PLOTLY_TEMPLATE, xaxis_tickangle=-45)
    fig_stack.show()
    save_figure(fig_stack, f"{stem}_stacked.html")

    sig = residuals.stack().reset_index()
    sig.columns = [x, y, "standardized_residual"]
    sig = sig[sig["standardized_residual"].abs() >= 1.96].copy()
    sig["pair"] = f"{x} × {y}"
    residual_rows.append(sig)

association_results = pd.DataFrame(association_rows)
standardized_residual_findings = pd.concat(residual_rows, ignore_index=True) if residual_rows else pd.DataFrame()
display(association_results)
display(standardized_residual_findings.head(100))


,x,y,n_complete,rows,columns,chi2,dof,p_value,cramers_v,expected_below_5,expected_below_5_pct,expected_below_1,assumptions_ok,aggregated_to_other,title,status
0,violacao,faixa_etaria_da_vitima,7822,15,15,1342.226612,196,6.651514e-170,0.102399,4,1.78,0,True,"violacao, faixa_etaria_da_vitima",Violation by victim age group,completed
1,violacao,genero_da_vitima,6757,15,5,523.740295,56,3.712159e-77,0.131589,14,18.67,3,False,violacao,Violation by victim gender,completed
2,violacao,raca_cor_da_vitima,4791,15,6,670.493516,70,6.961373e-99,0.158408,1,1.11,0,True,violacao,Violation by victim race/color,completed
3,violacao,grupo_vulneravel,8211,15,9,852.661296,112,2.748815e-114,0.106237,56,41.48,14,False,violacao,Violation by vulnerable group,completed
4,violacao,relacao_vitima_suspeito,7901,15,15,1183.387632,196,1.043517e-140,0.094562,62,27.56,0,False,"violacao, relacao_vitima_suspeito",Violation by victim-suspect relationship,completed
5,violacao,denuncia_emergencial,8211,15,5,245.073170,56,1.722172e-25,0.075890,40,53.33,29,False,violacao,Violation by emergency classification,completed
6,violacao,uf,7638,15,15,931.347260,196,4.813060e-96,0.083001,11,4.89,0,True,"violacao, uf",Violation by complaint UF,completed
7,grupo_vulneravel,faixa_etaria_da_vitima,7926,9,15,8549.693347,112,0.000000e+00,0.364971,59,43.70,26,False,faixa_etaria_da_vitima,Vulnerable group by victim age,completed
8,relacao_vitima_suspeito,genero_da_vitima,6652,15,5,1619.384876,56,7.201309e-302,0.242469,21,28.00,8,False,relacao_vitima_suspeito,Relationship by victim gender,completed
9,relacao_vitima_suspeito,faixa_etaria_da_vitima,7668,15,15,9764.992776,196,0.000000e+00,0.298830,59,26.22,0,False,"relacao_vitima_suspeito, faixa_etaria_da_vitima",Relationship by victim age,completed


,violacao,faixa_etaria_da_vitima,standardized_residual,pair,genero_da_vitima,raca_cor_da_vitima,grupo_vulneravel,relacao_vitima_suspeito,denuncia_emergencial,uf,genero_do_suspeito,faixa_etaria_do_suspeito,cenario_da_violacao,frequencia,canal_de_atendimento
0,INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE,13 ANOS,-3.231591,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE,18 A 19 ANOS,2.181146,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE,20 A 24 ANOS,-2.239105,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE,30 A 34 ANOS,-2.660591,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE,CRIANÇA/ADOLESCENTE IDADE NÃO INFORMADA,2.208739,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,INTEGRIDADE>FÍSICA>MAUS TRATOS,13 ANOS,-2.999811,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,INTEGRIDADE>FÍSICA>MAUS TRATOS,15 ANOS,2.031276,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,INTEGRIDADE>FÍSICA>MAUS TRATOS,18 A 19 ANOS,-2.711267,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,INTEGRIDADE>FÍSICA>MAUS TRATOS,45 A 49 ANOS,4.221906,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,INTEGRIDADE>FÍSICA>OUTROS,13 ANOS,-2.669659,violacao × faixa_etaria_da_vitima,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## 5.6 Correct for multiple testing

Because several hypotheses are evaluated, raw p-values are adjusted using the Benjamini-Hochberg false discovery rate. Associations should be interpreted using the adjusted p-value, effect size, expected-cell diagnostics, and substantive relevance together.


In [6]:
def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * n / np.arange(1, n+1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    result = np.empty(n)
    result[order] = adjusted
    return result

completed = association_results[association_results["status"] == "completed"].copy()
if not completed.empty:
    completed["p_fdr_bh"] = benjamini_hochberg(completed["p_value"])
    completed["significant_fdr"] = completed["p_fdr_bh"] < ALPHA
    completed["effect_interpretation"] = pd.cut(
        completed["cramers_v"],
        bins=[-np.inf, .10, .30, .50, np.inf],
        labels=["negligible", "small", "moderate", "large"],
        right=False
    ).astype("string")
    completed["reportable_inference"] = completed["significant_fdr"] & completed["assumptions_ok"]
    association_results = association_results.drop(columns=[c for c in ["p_fdr_bh", "significant_fdr", "effect_interpretation", "reportable_inference"] if c in association_results])
    association_results = association_results.merge(
        completed[["x", "y", "p_fdr_bh", "significant_fdr", "effect_interpretation", "reportable_inference"]],
        on=["x", "y"], how="left"
    )

display(association_results.sort_values("cramers_v", ascending=False))


,x,y,n_complete,rows,columns,chi2,dof,p_value,cramers_v,expected_below_5,expected_below_5_pct,expected_below_1,assumptions_ok,aggregated_to_other,title,status,p_fdr_bh,significant_fdr,effect_interpretation,reportable_inference
14,canal_de_atendimento,denuncia_emergencial,8401,8,5,8514.311985,28,0.000000e+00,0.502652,26,65.00,21,False,,Channel by emergency status,completed,0.000000e+00,True,large,False
7,grupo_vulneravel,faixa_etaria_da_vitima,7926,9,15,8549.693347,112,0.000000e+00,0.364971,59,43.70,26,False,faixa_etaria_da_vitima,Vulnerable group by victim age,completed,0.000000e+00,True,moderate,False
9,relacao_vitima_suspeito,faixa_etaria_da_vitima,7668,15,15,9764.992776,196,0.000000e+00,0.298830,59,26.22,0,False,"relacao_vitima_suspeito, faixa_etaria_da_vitima",Relationship by victim age,completed,0.000000e+00,True,small,False
12,denuncia_emergencial,cenario_da_violacao,8401,5,15,2212.309688,56,0.000000e+00,0.253374,40,53.33,28,False,cenario_da_violacao,Emergency status by setting,completed,0.000000e+00,True,small,False
8,relacao_vitima_suspeito,genero_da_vitima,6652,15,5,1619.384876,56,7.201309e-302,0.242469,21,28.00,8,False,relacao_vitima_suspeito,Relationship by victim gender,completed,1.800327e-301,True,small,False
10,genero_da_vitima,genero_do_suspeito,5660,5,5,1260.944590,16,1.228666e-258,0.234579,7,28.00,2,False,,Victim gender by suspect gender,completed,2.303749e-258,True,small,False
11,faixa_etaria_da_vitima,faixa_etaria_do_suspeito,4141,15,15,2865.290216,196,0.000000e+00,0.214938,61,27.11,10,False,"faixa_etaria_da_vitima, faixa_etaria_do_suspeito",Victim age by suspect age,completed,0.000000e+00,True,small,False
13,denuncia_emergencial,frequencia,8401,5,7,1338.463429,24,6.978017e-268,0.197825,18,51.43,14,False,,Emergency status by reported frequency,completed,1.495289e-267,True,small,False
2,violacao,raca_cor_da_vitima,4791,15,6,670.493516,70,6.961373e-99,0.158408,1,1.11,0,True,violacao,Violation by victim race/color,completed,8.701716e-99,True,small,True
1,violacao,genero_da_vitima,6757,15,5,523.740295,56,3.712159e-77,0.131589,14,18.67,3,False,violacao,Violation by victim gender,completed,3.977313e-77,True,small,False


### Effect-size overview

Cramér's V is used as a descriptive strength indicator. The cutoffs shown are broad heuristics and should not replace context, table dimensions, or practical relevance.


In [7]:
effect_plot = association_results[association_results["status"] == "completed"].copy()
if not effect_plot.empty:
    effect_plot["pair"] = effect_plot["x"] + " × " + effect_plot["y"]
    effect_plot = effect_plot.sort_values("cramers_v")
    fig = px.bar(
        effect_plot, x="cramers_v", y="pair", orientation="h",
        color="effect_interpretation", text="cramers_v",
        hover_data=["p_value", "p_fdr_bh", "assumptions_ok", "n_complete"],
        title="Strength of categorical associations",
        height=max(500, len(effect_plot)*36)
    )
    fig.add_vline(x=.10, line_dash="dot", line_color="gray")
    fig.add_vline(x=.30, line_dash="dot", line_color="gray")
    fig.add_vline(x=.50, line_dash="dot", line_color="gray")
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    save_figure(fig, "categorical_effect_sizes.html")


## 5.7 Sparse-table and disclosure-risk review

Small observed cells can create unstable percentages and disclosure risk. Cells below 5 are listed for review. Do not publish detailed combinations that could identify individuals or very small groups.


In [8]:
small_cell_rows = []
for x, y, title in available_pairs:
    pair, _ = prepare_pair(df_analysis, x, y)
    table = pd.crosstab(pair[x], pair[y])
    if table.empty:
        continue
    stacked = table.stack().reset_index(name="count")
    stacked = stacked[(stacked["count"] > 0) & (stacked["count"] < 5)]
    for _, row in stacked.iterrows():
        small_cell_rows.append({
            "pair": f"{x} × {y}", "row_category": row[x],
            "column_category": row[y], "count": int(row["count"])
        })
small_cells = pd.DataFrame(small_cell_rows)
display(small_cells.head(100))


,pair,row_category,column_category,count
0,violacao × faixa_etaria_da_vitima,INTEGRIDADE>FÍSICA>MAUS TRATOS,13 ANOS,1
1,violacao × faixa_etaria_da_vitima,INTEGRIDADE>FÍSICA>MAUS TRATOS,18 A 19 ANOS,1
2,violacao × faixa_etaria_da_vitima,INTEGRIDADE>FÍSICA>OUTROS,12 ANOS,3
3,violacao × faixa_etaria_da_vitima,INTEGRIDADE>FÍSICA>OUTROS,14 ANOS,4
4,violacao × faixa_etaria_da_vitima,INTEGRIDADE>FÍSICA>OUTROS,16 ANOS,1
5,violacao × faixa_etaria_da_vitima,INTEGRIDADE>NEGLIGÊNCIA,13 ANOS,3
6,violacao × faixa_etaria_da_vitima,INTEGRIDADE>NEGLIGÊNCIA,45 A 49 ANOS,4
7,violacao × faixa_etaria_da_vitima,INTEGRIDADE>PSÍQUICA>AMEAÇA OU COAÇÃO,17 ANOS,2
8,violacao × faixa_etaria_da_vitima,INTEGRIDADE>PSÍQUICA>AMEAÇA OU COAÇÃO,MENOS DE 01 ANO,1
9,violacao × faixa_etaria_da_vitima,INTEGRIDADE>PSÍQUICA>CONSTRANGIMENTO,17 ANOS,4


## 5.8 Numeric outcome by categorical groups

`sl_quantidade_vitimas` is often right-skewed and may contain legitimate collective complaints. Therefore, the notebook uses medians, interquartile ranges, Plotly boxplots, Mann-Whitney U for two groups, and Kruskal-Wallis for three or more groups. Statistical significance does not by itself imply a meaningful difference.


In [9]:
NUMERIC_OUTCOME = "sl_quantidade_vitimas"
NUMERIC_GROUPS = [
    "denuncia_emergencial", "grupo_vulneravel", "violacao",
    "cenario_da_violacao", "canal_de_atendimento", "uf"
]
numeric_test_rows = []

if NUMERIC_OUTCOME in df_analysis.columns:
    for group in [c for c in NUMERIC_GROUPS if c in df_analysis.columns]:
        pair = df_analysis[[group, NUMERIC_OUTCOME]].dropna().copy()
        pair[NUMERIC_OUTCOME] = pd.to_numeric(pair[NUMERIC_OUTCOME], errors="coerce")
        pair = pair.dropna()
        if pair[group].nunique() > MAX_LEVELS:
            keep = top_levels(pair[group], MAX_LEVELS-1)
            pair[group] = pair[group].where(pair[group].isin(keep), "OTHER")

        summary = pair.groupby(group)[NUMERIC_OUTCOME].agg(
            n="count", mean="mean", median="median", std="std",
            q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75), maximum="max"
        ).reset_index()
        summary.to_csv(TABLE_DIR / f"numeric_{safe_filename(group)}_summary.csv", index=False, encoding="utf-8-sig")
        display(Markdown(f"### `{NUMERIC_OUTCOME}` by `{group}`"))
        display(summary)

        fig = px.box(
            pair, x=group, y=NUMERIC_OUTCOME, points="outliers",
            title=f"Victim quantity by {group}",
            labels={NUMERIC_OUTCOME: "Victims per record"}
        )
        fig.update_layout(template=PLOTLY_TEMPLATE, xaxis_tickangle=-45)
        fig.show()
        save_figure(fig, f"numeric_{safe_filename(group)}_boxplot.html")

        groups = [g[NUMERIC_OUTCOME].to_numpy() for _, g in pair.groupby(group) if len(g) > 0]
        if len(groups) == 2:
            stat, p = mannwhitneyu(groups[0], groups[1], alternative="two-sided")
            test = "Mann-Whitney U"
        elif len(groups) >= 3:
            stat, p = kruskal(*groups)
            test = "Kruskal-Wallis"
        else:
            continue
        numeric_test_rows.append({
            "outcome": NUMERIC_OUTCOME, "group": group, "test": test,
            "groups": len(groups), "n_complete": len(pair),
            "statistic": stat, "p_value": p
        })

numeric_test_results = pd.DataFrame(numeric_test_rows)
if not numeric_test_results.empty:
    numeric_test_results["p_fdr_bh"] = benjamini_hochberg(numeric_test_results["p_value"])
    numeric_test_results["significant_fdr"] = numeric_test_results["p_fdr_bh"] < ALPHA
display(numeric_test_results)


### `sl_quantidade_vitimas` by `denuncia_emergencial`

,denuncia_emergencial,n,mean,median,std,q1,q3,maximum
0,N/D,1,10.0,10.0,<NA>,10.0,10.0,10
1,NÃO,7646,3.226916,1.0,7.676021,1.0,2.0,60
2,RISCO IMINENTE DE MORTE DA VÍTIMA,634,2.763407,1.0,3.603253,1.0,2.0,20
3,SITUAÇÃO FLAGRANTE - ATÉ 24H DA OCORRÊNCIA - SEM FREQUÊNCIA,115,10.26087,10.0,5.80243,10.0,10.0,20
4,VÍTIMA EM SANGRAMENTO,5,1.0,1.0,0.0,1.0,1.0,1


### `sl_quantidade_vitimas` by `grupo_vulneravel`

,grupo_vulneravel,n,mean,median,std,q1,q3,maximum
0,VIOLAÇÃO DE DIREITOS HUMANOS,105,22.952381,10.0,18.806904,10.0,50.0,50
1,VIOLÊNCIA CONTRA A MULHER,1112,1.579137,2.0,0.49392,1.0,2.0,2
2,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",1397,1.290623,1.0,0.74172,1.0,1.0,7
3,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,5202,3.766628,1.0,8.408421,1.0,2.0,60
4,VIOLÊNCIA CONTRA PESSOA COM DEFICIÊNCIA,237,1.848101,1.0,3.021485,1.0,2.0,20
5,VIOLÊNCIA CONTRA PESSOA EM SITUAÇÃO DE RUA,16,8.875,10.0,3.074085,10.0,10.0,10
6,VIOLÊNCIA CONTRA PESSOA IDOSA,49,4.204082,2.0,4.09766,1.0,10.0,10
7,VIOLÊNCIA CONTRA PESSOA SOCIALMENTE VULNERÁVEL,90,11.111111,10.0,3.160303,10.0,10.0,20
8,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,193,1.404145,1.0,2.285002,1.0,1.0,30


### `sl_quantidade_vitimas` by `violacao`

,violacao,n,mean,median,std,q1,q3,maximum
0,INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE,505,2.263366,1.0,4.03899,1.0,2.0,50
1,INTEGRIDADE>FÍSICA>MAUS TRATOS,292,2.760274,1.0,4.775805,1.0,2.0,40
2,INTEGRIDADE>FÍSICA>OUTROS,184,1.657609,1.0,1.481082,1.0,2.0,10
3,INTEGRIDADE>NEGLIGÊNCIA,496,1.252016,1.0,0.556863,1.0,1.0,7
4,INTEGRIDADE>PSÍQUICA>AMEAÇA OU COAÇÃO,341,3.228739,1.0,6.210086,1.0,2.0,50
5,INTEGRIDADE>PSÍQUICA>CONSTRANGIMENTO,332,2.373494,1.0,3.655764,1.0,2.0,30
6,INTEGRIDADE>PSÍQUICA>EXPOSIÇÃO,295,2.677966,1.0,4.667301,1.0,2.0,30
7,INTEGRIDADE>PSÍQUICA>TORTURA PSÍQUICA,535,2.102804,1.0,3.246821,1.0,2.0,30
8,LIBERDADE>DIREITOS INDIVIDUAIS>SEQUESTRO,238,3.987395,2.0,10.363634,1.0,2.0,60
9,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE CRIANÇAS - INTERNACIONAL,185,1.410811,1.0,0.662581,1.0,2.0,3


... (3 outputs left out)

,canal_de_atendimento,n,mean,median,std,q1,q3,maximum
0,CARTA,1,10.0,10.0,<NA>,10.0,10.0,10
1,E-MAIL,641,2.152886,2.0,1.516069,2.0,2.0,10
2,PORTAL,29,1.0,1.0,0.0,1.0,1.0,1
3,PROCESSO SEI,84,1.285714,1.0,0.454467,1.0,2.0,2
4,TELEFÔNICO,6091,3.259399,1.0,6.745724,1.0,2.0,60
5,TELEGRAM,14,4.857143,1.0,4.621973,1.0,10.0,10
6,WEBCHAT,46,10.347826,10.0,5.121707,10.0,10.0,20
7,WHATSAPP,1495,3.810033,1.0,11.103928,1.0,2.0,60


### `sl_quantidade_vitimas` by `uf`

,uf,n,mean,median,std,q1,q3,maximum
0,AM,210,3.057143,1.0,4.447092,1.0,2.0,20
1,BA,287,2.494774,1.0,4.281094,1.0,2.0,20
2,CE,325,12.326154,1.0,22.651236,1.0,2.0,60
3,DENUNCIANTE NÃO SOUBE INFORMAR,773,1.664942,1.0,1.185086,1.0,2.0,10
4,DF,177,1.378531,1.0,1.760744,1.0,1.0,10
5,GO,200,5.535,1.0,9.316801,1.0,4.0,40
6,MG,975,4.133333,2.0,8.851192,1.0,2.0,60
7,MS,191,1.801047,1.0,2.569497,1.0,1.0,10
8,OTHER,1068,3.326779,1.0,4.905953,1.0,2.0,30
9,PE,231,2.134199,1.0,2.533978,1.0,2.0,10


,outcome,group,test,groups,n_complete,statistic,p_value,p_fdr_bh,significant_fdr
0,sl_quantidade_vitimas,denuncia_emergencial,Kruskal-Wallis,5,8401,232.864175,3.191345e-49,3.829614e-49,True
1,sl_quantidade_vitimas,grupo_vulneravel,Kruskal-Wallis,9,8401,923.946772,3.855055e-194,2.313033e-193,True
2,sl_quantidade_vitimas,violacao,Kruskal-Wallis,15,8211,161.311813,3.865334e-27,3.865334e-27,True
3,sl_quantidade_vitimas,cenario_da_violacao,Kruskal-Wallis,15,8401,622.229571,9.844336e-124,1.968867e-123,True
4,sl_quantidade_vitimas,canal_de_atendimento,Kruskal-Wallis,8,8401,515.978766,2.939514e-107,4.409271e-107,True
5,sl_quantidade_vitimas,uf,Kruskal-Wallis,15,7828,623.668627,4.860706e-124,1.458212e-123,True


## 5.9 Missingness by subgroup

Group-specific missingness can bias comparisons. This section checks whether key victim and suspect variables are recorded at different rates across complaint channel, UF, vulnerable group, and emergency classification.


In [10]:
MISSINGNESS_TARGETS = [
    "genero_da_vitima", "faixa_etaria_da_vitima", "raca_cor_da_vitima",
    "genero_do_suspeito", "faixa_etaria_do_suspeito", "raca_cor_do_suspeito",
    "motivacao"
]
MISSINGNESS_GROUPS = ["canal_de_atendimento", "uf", "grupo_vulneravel", "denuncia_emergencial"]
missingness_group_rows = []

for target in [c for c in MISSINGNESS_TARGETS if c in df_analysis.columns]:
    for group in [c for c in MISSINGNESS_GROUPS if c in df_analysis.columns]:
        temp = df_analysis[[group, target]].copy()
        temp["is_missing"] = temp[target].isna()
        summary = temp.dropna(subset=[group]).groupby(group)["is_missing"].agg(["size", "sum", "mean"]).reset_index()
        summary["missing_pct"] = (summary["mean"]*100).round(2)
        for _, row in summary.iterrows():
            missingness_group_rows.append({
                "target": target, "group_variable": group,
                "group_category": row[group], "n": int(row["size"]),
                "missing_n": int(row["sum"]), "missing_pct": row["missing_pct"]
            })

missingness_by_group = pd.DataFrame(missingness_group_rows)
display(missingness_by_group.head(100))


,target,group_variable,group_category,n,missing_n,missing_pct
0,genero_da_vitima,canal_de_atendimento,CARTA,1,0,0.00
1,genero_da_vitima,canal_de_atendimento,E-MAIL,641,39,6.08
2,genero_da_vitima,canal_de_atendimento,PORTAL,29,7,24.14
3,genero_da_vitima,canal_de_atendimento,PROCESSO SEI,84,33,39.29
4,genero_da_vitima,canal_de_atendimento,TELEFÔNICO,6091,1014,16.65
5,genero_da_vitima,canal_de_atendimento,TELEGRAM,14,6,42.86
6,genero_da_vitima,canal_de_atendimento,WEBCHAT,46,3,6.52
7,genero_da_vitima,canal_de_atendimento,WHATSAPP,1495,352,23.55
8,genero_da_vitima,uf,AC,93,81,87.10
9,genero_da_vitima,uf,AL,10,0,0.00


## 5.10 Findings register

The automated register prioritizes associations with adequate assumptions, FDR-adjusted significance, and non-negligible effect size. Residuals identify which category combinations contribute most strongly, but they must be interpreted with suppression and multiplicity considerations.


In [11]:
finding_rows = []
for _, row in association_results[association_results["status"] == "completed"].iterrows():
    if not row.get("assumptions_ok", False):
        finding_rows.append({
            "pair": f"{row['x']} × {row['y']}", "finding_type": "sparse_table",
            "finding": f"{row['expected_below_5_pct']:.2f}% of expected cells below 5",
            "interpretation": "Do not rely on asymptotic chi-square inference; review category aggregation or an exact/Monte Carlo approach."
        })
    elif row.get("significant_fdr", False):
        finding_rows.append({
            "pair": f"{row['x']} × {row['y']}", "finding_type": "association",
            "finding": f"FDR p={row['p_fdr_bh']:.4g}; Cramér's V={row['cramers_v']:.3f} ({row['effect_interpretation']})",
            "interpretation": "Describe the association and inspect standardized residuals; avoid causal language."
        })

bivariate_findings = pd.DataFrame(finding_rows)
display(bivariate_findings)


,pair,finding_type,finding,interpretation
0,violacao × faixa_etaria_da_vitima,association,FDR p=1.109e-169; Cramér's V=0.102 (small),Describe the association and inspect standardized residuals; avoid causal language.
1,violacao × genero_da_vitima,sparse_table,18.67% of expected cells below 5,Do not rely on asymptotic chi-square inference; review category aggregation or an exact/Monte Carlo approach.
2,violacao × raca_cor_da_vitima,association,FDR p=8.702e-99; Cramér's V=0.158 (small),Describe the association and inspect standardized residuals; avoid causal language.
3,violacao × grupo_vulneravel,sparse_table,41.48% of expected cells below 5,Do not rely on asymptotic chi-square inference; review category aggregation or an exact/Monte Carlo approach.
4,violacao × relacao_vitima_suspeito,sparse_table,27.56% of expected cells below 5,Do not rely on asymptotic chi-square inference; review category aggregation or an exact/Monte Carlo approach.
5,violacao × denuncia_emergencial,sparse_table,53.33% of expected cells below 5,Do not rely on asymptotic chi-square inference; review category aggregation or an exact/Monte Carlo approach.
6,violacao × uf,association,FDR p=5.554e-96; Cramér's V=0.083 (negligible),Describe the association and inspect standardized residuals; avoid causal language.
7,grupo_vulneravel × faixa_etaria_da_vitima,sparse_table,43.70% of expected cells below 5,Do not rely on asymptotic chi-square inference; review category aggregation or an exact/Monte Carlo approach.
8,relacao_vitima_suspeito × genero_da_vitima,sparse_table,28.00% of expected cells below 5,Do not rely on asymptotic chi-square inference; review category aggregation or an exact/Monte Carlo approach.
9,relacao_vitima_suspeito × faixa_etaria_da_vitima,sparse_table,26.22% of expected cells below 5,Do not rely on asymptotic chi-square inference; review category aggregation or an exact/Monte Carlo approach.


## 5.11 Export results and metadata


In [12]:
association_results.to_csv(TABLE_DIR / "categorical_association_results.csv", index=False, encoding="utf-8-sig")
standardized_residual_findings.to_csv(TABLE_DIR / "standardized_residual_findings.csv", index=False, encoding="utf-8-sig")
small_cells.to_csv(TABLE_DIR / "small_cells_under_5.csv", index=False, encoding="utf-8-sig")
numeric_test_results.to_csv(TABLE_DIR / "numeric_group_test_results.csv", index=False, encoding="utf-8-sig")
missingness_by_group.to_csv(TABLE_DIR / "missingness_by_group.csv", index=False, encoding="utf-8-sig")
bivariate_findings.to_csv(TABLE_DIR / "bivariate_findings_register.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(available_pairs, columns=["x", "y", "title"]).to_csv(TABLE_DIR / "available_pair_plan.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(skipped_pairs, columns=["x", "y", "title"]).to_csv(TABLE_DIR / "skipped_pair_plan.csv", index=False, encoding="utf-8-sig")

figure_index = pd.DataFrame({"figure_file": sorted(p.name for p in FIGURE_DIR.glob("*.html"))})
figure_index.to_csv(OUTPUT_DIR / "figure_index.csv", index=False, encoding="utf-8-sig")

metadata = {
    "data_source": str(DATA_SOURCE), "counting_mode": COUNTING_MODE,
    "analytical_rows": int(len(df_analysis)), "identifier": IDENTIFIER,
    "alpha": ALPHA, "minimum_expected_cell": MIN_EXPECTED,
    "maximum_levels_per_variable": MAX_LEVELS,
    "multiple_testing_correction": "Benjamini-Hochberg FDR"
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Tables:", TABLE_DIR.resolve())
print("Figures:", FIGURE_DIR.resolve())


Tables: C:\Projetos\eda-disque100-completo\eda\outputs\step_05_bivariate_analysis\tables
Figures: C:\Projetos\eda-disque100-completo\eda\outputs\step_05_bivariate_analysis\figures


## 5.12 Interpretation checklist

Before reporting a bivariate result:

1. Confirm the analytical unit and denominator.
2. Report complete-case `n` for the pair.
3. Present counts plus row or column percentages.
4. Check expected-cell diagnostics.
5. Use the Benjamini-Hochberg-adjusted p-value when comparing multiple pairs.
6. Report Cramér's V, not only statistical significance.
7. Inspect standardized residuals to locate category combinations driving the association.
8. Suppress or aggregate cells with fewer than 5 observations when disclosure risk exists.
9. Examine subgroup missingness before interpreting profile differences.
10. Avoid causal, prevalence, and legal-responsibility claims.

### Suggested reporting pattern

“Among complete recorded observations, violation category was associated with victim age group, χ²(df) = value, FDR-adjusted p = value, Cramér's V = value. Standardized residuals indicated that selected violation-age combinations occurred more or less frequently than expected under independence. Results describe recorded hotline observations and may reflect reporting and data-collection processes.”

## Next step

**Step 6: Temporal Analysis** should investigate monthly trends, seasonality, structural breaks, campaign or policy periods, and changes in data completeness over time. It should distinguish changes in reporting volume from changes in the underlying occurrence of violations.
